# Anima + WAI-Anima — ComfyUI Colab

Один ноутбук для Anima Aesthetic v1.1 и WAI-Anima v1.0. Устанавливает ComfyUI, ComfyUI-Manager, Anima-LLLite и готовые T2I/ControlNet/Inpaint workflow. Токены берутся из Colab Secrets (`HF_TOKEN`, `CIVITAI_API_TOKEN`) или запрашиваются интерактивно.

In [ ]:
# @title 1) Tokens and paths
import os, getpass
from pathlib import Path
try:
    from google.colab import userdata
except Exception:
    userdata = None

def secret(name):
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if isinstance(value, dict):
        value = value.get('value') or value.get('token') or next(iter(value.values()), '')
    if not isinstance(value, str):
        value = str(value) if value else ''
    return value.strip() or os.environ.get(name, '').strip()

HF_TOKEN = secret('HF_TOKEN') or secret('HUGGINGFACE_TOKEN')
CIVITAI_API_TOKEN = secret('CIVITAI_API_TOKEN')
if not HF_TOKEN: HF_TOKEN = getpass.getpass('Hugging Face token (required): ').strip()
if not CIVITAI_API_TOKEN: CIVITAI_API_TOKEN = getpass.getpass('Civitai API token (recommended): ').strip()
if not HF_TOKEN: raise RuntimeError('HF_TOKEN is required.')
os.environ.update({'HF_TOKEN': HF_TOKEN, 'HUGGINGFACE_TOKEN': HF_TOKEN})
if CIVITAI_API_TOKEN: os.environ['CIVITAI_API_TOKEN'] = CIVITAI_API_TOKEN
COMFY_ROOT = Path('/content/ComfyUI'); MODEL_ROOT = COMFY_ROOT / 'models'
print('Tokens configured without displaying their values.')


In [ ]:
# @title 2) Install ComfyUI, Manager and Anima nodes
import subprocess
def run(cmd):
    print('+', cmd); subprocess.run(cmd, shell=True, check=True)
if not COMFY_ROOT.exists(): run('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI')
run('pip install -q -r /content/ComfyUI/requirements.txt')
NODES = {'ComfyUI-Manager':'https://github.com/ltdrdata/ComfyUI-Manager.git','ComfyUI-Workflow-Models-Downloader':'https://github.com/slahiri/ComfyUI-Workflow-Models-Downloader.git','ComfyUI-Anima-LLLite':'https://github.com/kohya-ss/ComfyUI-Anima-LLLite.git','comfyui_controlnet_aux':'https://github.com/Fannovel16/comfyui_controlnet_aux.git','comfyui-lora-manager':'https://github.com/willmiao/ComfyUI-Lora-Manager.git','rgthree-comfy':'https://github.com/rgthree/rgthree-comfy.git','was-node-suite-comfyui':'https://github.com/WASasquatch/was-node-suite-comfyui.git','ComfyUI-Image-Saver':'https://github.com/alexopus/ComfyUI-Image-Saver.git'}
for folder, repo in NODES.items():
    target = COMFY_ROOT/'custom_nodes'/folder
    if not target.exists(): run(f'git clone --depth 1 {repo} {target}')
    req = target/'requirements.txt'
    if req.exists(): run(f'pip install -q -r {req}')
print('ComfyUI and Anima node set are ready.')

In [ ]:
# @title 3) Download both Anima checkpoints and dependencies
import requests
def download(url, target, headers=None):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>1024: print('exists:',target); return
    h={'Authorization':f'Bearer {HF_TOKEN}'} if 'huggingface.co' in url else {}
    if headers: h.update(headers)
    with requests.get(url,headers=h,stream=True,timeout=60) as r:
        r.raise_for_status()
        with open(target,'wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk: f.write(chunk)
    print('downloaded:',target)
def civitai(url,target):
    h={'Authorization':f'Bearer {CIVITAI_API_TOKEN}'} if CIVITAI_API_TOKEN else {}
    download(url,target,h)
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors',MODEL_ROOT/'text_encoders/qwen_3_06b_base.safetensors')
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors',MODEL_ROOT/'vae/qwen_image_vae.safetensors')
download('https://huggingface.co/Comfy-Org/Anima-LLLite/resolve/main/model_patches/anima-lllite-any-test-like-v2.safetensors',MODEL_ROOT/'model_patches/anima-lllite-any-test-like-v2.safetensors')
download('https://huggingface.co/Comfy-Org/Anima-LLLite/resolve/main/model_patches/anima-lllite-inpainting-v2.safetensors',MODEL_ROOT/'model_patches/anima-lllite-inpainting-v2.safetensors')
civitai('https://civitai.red/api/download/models/3126581?fileId=3007030',MODEL_ROOT/'diffusion_models/anima/anima_aestheticV11.safetensors')
civitai('https://civitai.red/api/download/models/2983680?fileId=2863158',MODEL_ROOT/'diffusion_models/anima/waiANIMA_v10Base10.safetensors')
print('Both checkpoints are available in ComfyUI.')

In [ ]:
# @title 5) Launch ComfyUI + resilient Cloudflare tunnel
import os, re, shutil, socket, subprocess, time, requests, threading
from pathlib import Path

LOW_VRAM_STABLE = False
COMFY_ROOT = Path(COMFY_ROOT)
output_dir = COMFY_ROOT / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

comfy_args = [
    'python', 'main.py', '--listen', '0.0.0.0', '--port', '8188',
    '--enable-cors-header', '*',
    '--output-directory', str(output_dir),
]
comfy_args += (
    ['--novram', '--disable-smart-memory', '--cache-none', '--force-upcast-attention']
    if LOW_VRAM_STABLE else ['--lowvram', '--preview-method', 'auto']
)
comfy = subprocess.Popen(comfy_args, cwd=COMFY_ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

def wait_port(host='127.0.0.1', port=8188, timeout=240):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with socket.create_connection((host, port), timeout=2):
                return True
        except OSError:
            if comfy.poll() is not None:
                raise RuntimeError('ComfyUI exited before opening port 8188.')
            time.sleep(2)
    return False

if not wait_port():
    raise TimeoutError('ComfyUI port 8188 did not open.')

def tunnel_is_alive(url):
    for path in ('/system_stats', '/', '/object_info'):
        try:
            r = requests.get(url.rstrip('/') + path, timeout=12)
            if 200 <= r.status_code < 300:
                return True
        except requests.RequestException:
            pass
    return False

def stop_tunnel(proc):
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()

def create_tunnel():
    protocols = ('http2', 'quic')
    attempt = 0
    while True:
        protocol = protocols[attempt % len(protocols)]
        attempt += 1
        print(f'Tunnel attempt {attempt}: protocol={protocol}')
        proc = subprocess.Popen(
            [
                'cloudflared', 'tunnel', '--no-autoupdate',
                '--url', 'http://127.0.0.1:8188',
                '--protocol', protocol, '--edge-ip-version', '4',
                '--loglevel', 'info',
            ],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        url = None
        recent = []
        deadline = time.time() + 90
        while time.time() < deadline:
            line = proc.stdout.readline()
            if line:
                line = line.strip()
                recent = (recent + [line])[-8:]
                match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line, re.I)
                if match:
                    url = match.group(0)
                    break
            elif proc.poll() is not None:
                break
        if url:
            print('Tunnel URL found. Waiting for stable propagation...')
            ready_deadline = time.time() + 180
            good = 0
            while time.time() < ready_deadline and proc.poll() is None:
                if tunnel_is_alive(url):
                    good += 1
                    if good >= 3:
                        return proc, url
                else:
                    good = 0
                time.sleep(5)
        print('Tunnel attempt failed:', recent[-3:] or ['no cloudflared output'])
        stop_tunnel(proc)
        time.sleep(min(60, 10 + attempt * 5))

tunnel, url = create_tunnel()
print('Open ComfyUI:', url)

while True:
    time.sleep(20)
    if tunnel.poll() is not None or not tunnel_is_alive(url):
        print('Tunnel became unhealthy; recreating automatically...')
        stop_tunnel(tunnel)
        tunnel, url = create_tunnel()
        print('New ComfyUI URL:', url)

